# Fault and downtime analysis (diagnostic)

Supporting analysis for PTH Tier 1 metrics **ROSP-11** (time-loss accounting) and
**ROSP-12** (fault rate, MTTR and top fault categories) - {{ params.start_dayobs }} to
{{ params.end_dayobs }} (dayObs, inclusive).

This is a **diagnostic** notebook, not an official metric page. It develops the per-night
time attribution and fault statistics that both metrics depend on, as input to the time
attribution proposal in [ROSP-24](https://rubinobs.atlassian.net/browse/ROSP-24). The
official pages remain `rosp-11_time_loss_accounting` and `rosp-12_fault_rate_mttr`.

- Jira epics: [ROSP-11](https://rubinobs.atlassian.net/browse/ROSP-11),
  [ROSP-12](https://rubinobs.atlassian.net/browse/ROSP-12)
- Performance Tracking Hub: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)

Steward: Brian Stalder. Method: in development, not baselined; nothing here is a frozen
metric value.

**Aligned with the ROSP-24 loss-cause taxonomy v1.0 (pending sign-off), 2026-11-17.** The state reduction
follows the ROSP-24 precedence (weather masks faults and unplanned downtime; a fault outranks unplanned
`DOWNTIME`), override nights take their ROSP-24 `category`, the narrative log annotates but no longer sets
hours, and each fault run is charged to its primary CSC (the first to enter `FAULT`) in the ROSP-24
attribution groups rather than split over every CSC named. The official ledger, with the engineering (B2),
visit-gap (D3) and start-up (A3) splits this notebook does not make, is `rosp-11_time_loss_accounting`.

In [ ]:
# Parameters cell. Set defaults here - gets replaced by the sidecar on Times Square.
# Defaults are the PTH baseline epoch.
start_dayobs = 20260629  # First dayObs (inclusive)
end_dayobs = 20261231  # Last dayObs (inclusive)

# Note: the EFD observatoryStatus topic this notebook is built on only begins
# 20260225. Earlier nights have no state: they stay unmeasured (ROSP-24 F), the
# narrative log is shown for them as annotation only, and they are excluded from
# the per-component fault statistics.

not_times_square = True

In [ ]:
# Times Square runs this notebook in a Nublado pod where rubin_nights
# may be missing or stale; install/upgrade it there. When running
# anywhere else (e.g. an SDF login node) this cell is a no-op.
try:
    not_times_square
except NameError:
    not_times_square = False

if not not_times_square:
    print("updating rubin_nights")
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps > /dev/null 2>&1

# Import here rather than relying on the parameters cell above, which Times Square
# replaces wholesale with sidecar values (dropping any imports it contained).
import rubin_nights.dayobs_utils as rn_dayobs

# Times Square passes parameters as strings. The rest of the notebook wants ints:
# f-string SQL and str(day_obs) slicing tolerate strings, but rn_dayobs and the
# %i format specifiers below do not. day_obs_str_to_int accepts both "20260629"
# and the dashed "2026-06-29" that a `format: dayobs` parameter would arrive as.
start_dayobs = rn_dayobs.day_obs_str_to_int(str(start_dayobs))
end_dayobs = rn_dayobs.day_obs_str_to_int(str(end_dayobs))

# Analyse complete nights only. Nights after yesterday's dayObs have not been
# observed yet; left in, they add empty bars and are reported as "covered".
requested_end_dayobs = end_dayobs
end_dayobs = min(
    end_dayobs, rn_dayobs.day_obs_str_to_int(rn_dayobs.yesterday_day_obs())
)
if end_dayobs < requested_end_dayobs:
    print(
        f"Requested range ends {requested_end_dayobs}; analysing through "
        f"{end_dayobs}, the last complete night."
    )
if start_dayobs > end_dayobs:
    raise ValueError(f"start_dayobs {start_dayobs} is after {end_dayobs}")

In [ ]:
SAVE_FIGURES = False
# Survey programs, drawn as ticks on the state timeline so a gap can be read
# off against what the observatory was doing.
SURVEY_PROGRAMS = [
    "BLOCK-407",
    "BLOCK-408",
    "BLOCK-416",
    "BLOCK-417",
    "BLOCK-419",
    "BLOCK-421",
]  # Early Ops

# Fault and Downtime Analysis

Where the lost on-sky time went. A view built from the gaps *between consecutive visits*
is blind to the nights that cost the most: over a long range several nights have no LSSTCam
visits at all, and therefore no gaps. This notebook accounts for the whole night instead.
For every `day_obs` in the
`start_dayobs`–`end_dayobs` range it:

1. **Partitions the night** into operational time versus time lost to weather, downtime, a
   fault, or idling, from the EFD `lsst.sal.Scheduler.logevent_observatoryStatus` topic, so
   that the per-night hours sum exactly to the length of the $-12^\circ$ night.
2. **Attributes fault hours to a subsystem**: each fault run goes to its primary CSC (the first
   to enter `FAULT`, parsed from the scheduler's status notes) and that CSC's ROSP-24
   attribution group.
3. **Turns those faults into reliability statistics** — fault rate per hour of exposure,
   mean time between faults, and recovery time — per component, and trends them over the
   range.
4. **Cross-checks** the partition against the overlapping `rubin_nights` observatory states
   and the hand-written narrative log, and reconciles the differences.

`start_dayobs` / `end_dayobs` are supplied by the sidecar YAML when run as a Times Square
page; the parameters cell above sets local defaults and the `not_times_square` runtime flag.

Note that the EFD topic this notebook is built on begins **2026-02-25**. Over a range reaching
earlier than that, those nights stay unmeasured (ROSP-24 F), with the narrative log shown only as
annotation, and are excluded from the per-component statistics; the notebook detects the boundary and reports it
rather than assuming a fixed date. The same applies to **gaps** in the topic, such as the one
that opened after the July 2026 storm: a state is trusted for at most 30 hours after its
message, and longer silences are treated as unmeasured rather than carried forward. Periods
the EFD did not record, such as the July–October 2026 off-sky period, come from the shared
observatory state override file
([ROSP-42](https://rubinobs.atlassian.net/browse/ROSP-42)).

In [ ]:
import re

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from astropy.time import Time, TimeDelta
import astropy.units as u

from IPython.display import display

import rubin_nights.dayobs_utils as rn_dayobs
import rubin_nights.observatory_status as observatory_status

In [ ]:
from lsst.utils.plotting import publication_plots

publication_plots.set_rubin_plotstyle()

In [ ]:
# Connect via rubin_nights, which resolves the correct endpoints and RSP token
# for whatever environment this runs in (Times Square / Nublado in-cluster, the
# USDF RSP, or an SDF login node). This notebook needs three of them: the
# ConsDB for visits, the EFD for observatory status, and the narrative log.
from rubin_nights import connections

clients = connections.get_clients()  # site auto-detected, token auto-resolved
client = clients["consdb"]
efd_client = clients["efd"]

In [ ]:
# Visits are used only to place survey ticks on the state timeline and to find
# inter-visit gaps for the event table, so pull just the columns those need.
instrument = "lsstcam"


def fetch(start_dayobs, end_dayobs):
    visits_query = f"""
    SELECT
        day_obs, science_program, obs_start_mjd, obs_end_mjd, exp_midpt_mjd
    FROM
        cdb_{instrument}.visit1 AS visit1
    WHERE
        airmass > 0.
        AND img_type in ('acq', 'science', 'cwfs', 'focus')
        AND day_obs >= {start_dayobs} AND day_obs <= {end_dayobs}
    """
    return client.query(visits_query).sort_values(by="obs_start_mjd")


visits = fetch(start_dayobs, end_dayobs)
print(f"{len(visits)} visits on {visits.day_obs.nunique()} nights")

In [ ]:
# Time conventions for the state timeline below: the x-axis is the MJD of the
# day_obs at 0h UTC, and the y-axis is hours after the *following* 0h UTC.


def dayObsToTime(day_obs):
    year = str(day_obs)[0:4]
    month = str(day_obs)[4:6]
    day = str(day_obs)[6:8]
    time = Time("%s-%s-%s 00:00:00" % (year, month, day), format="iso")
    return time


def convertTime(time_input, time_reference):
    time_output = (time_input - (time_reference + (1.0 * u.day))) * u.day.to(u.hr)
    return time_output.value

## Building the lost-time partition

The source is the EFD topic `lsst.sal.Scheduler.logevent_observatoryStatus`, fetched via
`rubin_nights.observatory_status.get_obs_status_messages`. Each message carries
`statusLabels` (a pipe-joined set of `OPERATIONAL` / `IDLE` / `FAULT` / `WEATHER` /
`DOWNTIME` / `UNKNOWN`) and a `note` whose machine-generated clause names the faulting CSCs
("The following components are in `<State.FAULT: 3>` state: MTMount, MTPtg").

**Building an exclusive partition.** For each `day_obs` the state in force at $-12^\circ$
sunset is seeded from the last message at or before it; each subsequent message's state runs
until the next message or $-12^\circ$ sunrise. Because several labels can be set at once,
each interval is reduced to a single `cause` by the priority

`WEATHER > FAULT > DOWNTIME > IDLE > OPERATIONAL > UNKNOWN`

which is the [ROSP-24](https://rubinobs.atlassian.net/browse/ROSP-24) precedence
B1 → C → B2 → D (D2, D1, D4, D3) → E → A → F as far as the EFD states can express it: weather (C)
masks faults and unplanned downtime, a fault (D1/D2) outranks unplanned `DOWNTIME` (D4), and idle (E)
outranks operational (A). Planned downtime (B1) is not an EFD state and comes from the overrides.
`OPERATIONAL` holds science (A), engineering (B2) and visit-gap excess (D3) together; splitting them needs
visit metadata and the slew model and is done in the official `rosp-11_time_loss_accounting` notebook.

so the per-night hours sum to `night_hours` exactly (asserted below). This deliberately
differs from `observatory_status.get_observatory_state_times`, whose periods **overlap** and
cannot be summed — a `WEATHER` period typically spans the whole night while
`OPERATIONAL` / `IDLE` / `FAULT` run inside it. That function is still used at the end of the
notebook as an independent cross-check.

Weather outranks fault, so a fault occurring while the dome was already shut for weather is
charged to weather; those hours are the ROSP-24 shadow record, reported separately rather than
double counted. A fault during unplanned `DOWNTIME` is charged to the fault. Under ROSP-49 a fault
that covers a whole night with no survey visit (to within 30 minutes) is D2; every other fault is D1. Two further conventions to keep in mind when reading the numbers:

- `FAULT` hours include **intentional engineering faults** — for example, one note reads
  "We were intentionally faulting the mirror to perform some tests. Not an actual fault."
  on a message still labelled `FAULT`. Rather than filter on operator prose, the raw note is
  surfaced in the event table below.
- Each fault run is charged to its **primary CSC**, the first CSC to enter `FAULT` (ROSP-24),
  taken from the first current "are in" clause that describes the run: the run's first message;
  for a fault carried across twilight, the earliest clause of the unbroken `FAULT` stretch it
  continues; for a `FAULT` set by hand with no clause, the last clause within 24 hours before it;
  then later messages of the run, then the sticky "were in" clause. When the clause names several
  CSCs at once, their CSC `summaryState` transitions decide which faulted first. A cascade (the
  control system sends followers to `FAULT` after the root cause) therefore stays with the CSC that
  went first; `onset_source` and `cscs_named` in `fault_runs` show how each run was attributed. The
  attribution is to the CSC, not necessarily the root cause, and the operator note can disagree.
- `UNKNOWN` is kept as its own bucket rather than being bridged into the surrounding state
  (which is what `get_observatory_state_times` does), since a partition cannot relabel it.

**Telemetry gaps.** The Scheduler emits a message at least once a day while it is running,
so a state is trusted for at most `MAX_STATE_AGE_HOURS` (30 h) after its message. Beyond
that the time is recorded as unmeasured (`provenance = no_telemetry`) instead of inheriting
a state that may be weeks old. Without this, the last message before a long outage is
copied across every night of it. A night counts as **EFD-covered** only when at least half
of it has fresh telemetry, and only covered nights feed the fault statistics.

**State overrides.** Where the EFD state is missing or wrong, the shared override file kept
under [ROSP-42](https://rubinobs.atlassian.net/browse/ROSP-42)
(`tracking_hub/ops/observatory_state_overrides/observatory_state_overrides.csv`, read from
`main` at run time and pinned to the commit that last touched it, which is printed) gives the
state for whole nights. As its README specifies, an override replaces the EFD state for the
whole night. Each row carries its ROSP-24 `category`: B1 is `PLANNED_DOWNTIME`, C1 / C4 / C5 are
`WEATHER` (the category is kept in `override_category`), and D4 is `DOWNTIME`. Weather inside
planned downtime stays in `status_labels` and is reported as the shadow record rather than charged.
Overridden nights have no fresh telemetry, so they feed the hours budget but not the fault
statistics. If the file cannot be read, the notebook says so and applies no overrides.

In [ ]:
import json
import os
import urllib.request

# Observatory-state attribution: constants and helpers, aligned with the ROSP-24 loss-cause taxonomy
# v1.0 (pending sign-off; https://rubinobs.atlassian.net/browse/ROSP-24).

# Reduction order for a message with several labels set at once, highest first. This follows the ROSP-24
# precedence B1 -> C -> B2 -> D (D2, D1, D4, D3) -> E -> A -> F as far as the EFD states can express it:
# weather (C) masks faults and unplanned downtime, a fault (D1/D2) outranks unplanned DOWNTIME (D4), and
# idle (E) outranks operational (A). Planned downtime (B1) is not an EFD state; it comes from the overrides.
CAUSE_PRIORITY = ("WEATHER", "FAULT", "DOWNTIME", "IDLE", "OPERATIONAL", "UNKNOWN")

# Column and plotting order for the hours budget, in ROSP-24 precedence order.
BUDGET_CAUSES = (
    "PLANNED_DOWNTIME",
    "WEATHER",
    "FAULT",
    "DOWNTIME",
    "IDLE",
    "OPERATIONAL",
    "UNKNOWN",
)
LOST_CAUSES = ("PLANNED_DOWNTIME", "WEATHER", "FAULT", "DOWNTIME", "IDLE")

# What each budget cause is in the ROSP-24 taxonomy. OPERATIONAL holds the science time (A1-A3) together with
# engineering (B2) and visit-gap excess (D3); those need visit metadata and the slew model and are split in the
# official rosp-11_time_loss_accounting notebook, not here.
ROSP24_CATEGORY = {
    "PLANNED_DOWNTIME": "B1",
    "WEATHER": "C (C1-C3; C1/C4/C5 from overrides)",
    "FAULT": "D1 / D2 (whole-night fault, ROSP-49)",
    "DOWNTIME": "D4",
    "IDLE": "E",
    "OPERATIONAL": "A, B2 and D3 (not split here)",
    "UNKNOWN": "F",
}

# A status message's state is trusted for at most this long. The Scheduler emits
# at least one message a day while running, so a longer silence means it was not
# reporting, and carrying the old state forward would invent hours.
MAX_STATE_AGE_HOURS = 30.0
# A night is EFD-covered when at least this fraction of it has fresh telemetry.
MIN_COVERED_FRACTION = 0.5
# ROSP-49: D2 is a whole-night fault - a night with no survey visit whose FAULT state covers the night to within
# this many minutes. Every other FAULT is D1.
D2_TOLERANCE_MIN = 30.0
# A fault run whose own messages do not name a CSC takes the last one named this long before it.
FAULT_ONSET_LOOKBACK_HOURS = 24.0

# Shared observatory state overrides (ROSP-42), read from main at run time and pinned to the commit that last
# touched the file, so every Category 1 notebook applies the same record. Set ROSP_STATE_OVERRIDES_CSV (as in
# ROSP-9; STATE_OVERRIDES_URL is still accepted) to a local path or another URL to test changes to the file.
OVERRIDES_REPO = "lsst/sp_notebooks"
OVERRIDES_PATH = (
    "tracking_hub/ops/observatory_state_overrides/observatory_state_overrides.csv"
)
OVERRIDES_SOURCE = os.environ.get("ROSP_STATE_OVERRIDES_CSV") or os.environ.get(
    "STATE_OVERRIDES_URL"
)

CAUSE_COLORS = {
    "OPERATIONAL": "tab:blue",
    "FAULT": "tab:red",
    "WEATHER": "tab:cyan",
    "DOWNTIME": "tab:purple",
    "IDLE": "tab:orange",
    "UNKNOWN": "0.6",
    "PLANNED_DOWNTIME": "tab:pink",
}

# Faulting CSC -> ROSP-24 attribution group (the proposal in the ROSP-24 comments; Q5, the mapping check by the
# subsystem leads, is still open). Keyed on the *base* name so that SAL-indexed CSCs (MTHexapod:1, Scheduler:1)
# resolve via name.split(":")[0]. MTPtg is not in the ROSP-24 table and is grouped with the mount.
CSC_SUBSYSTEMS = {
    "MTMount": "Telescope mount and structure",
    "MTRotator": "Telescope mount and structure",
    "MTPtg": "Telescope mount and structure",
    "MTM1M3": "Optics and active optics",
    "MTM1M3TS": "Optics and active optics",
    "MTM2": "Optics and active optics",
    "MTHexapod": "Optics and active optics",
    "MTAOS": "Optics and active optics",
    "MTCamera": "Camera",
    "MTOODS": "Camera",
    "MTHeaderService": "Camera",
    "MTDome": "Dome and enclosure",
    "MTDomeTrajectory": "Dome and enclosure",
    "GIS": "Safety systems",
    "Scheduler": "Observatory control software",
    "ScriptQueue": "Observatory control software",
    "Watcher": "Observatory control software",
}
UNATTRIBUTED = "Pending attribution"

SUBSYSTEM_COLORS = {
    "Telescope mount and structure": "tab:red",
    "Optics and active optics": "tab:orange",
    "Camera": "tab:blue",
    "Dome and enclosure": "tab:purple",
    "Facility utilities": "tab:olive",
    "Safety systems": "tab:pink",
    "Observatory control software": "tab:brown",
    UNATTRIBUTED: "0.6",
}

# Match only the machine-generated clause, and only its *current* ("are in")
# form - "were in" is sticky and keeps repeating long after recovery. Operator
# free text may precede the clause and must not be searched.
FAULT_CSC_RE = re.compile(r"components are in <State\.FAULT: \d+> state:\s*([^.]*)\.")
# The sticky past-tense clause, used only as the last resort for naming a run's CSC.
FAULT_CSC_WERE_RE = re.compile(
    r"components were in <State\.FAULT: \d+> state:\s*([^.]*)\."
)


def fetch_status_messages(start_dayobs, end_dayobs, efd_client, lead_days=3):
    """Observatory status messages, with a lead-in so the state in force at the
    first sunset is known even when the last message predates the range."""
    messages = observatory_status.get_obs_status_messages(
        rn_dayobs.day_obs_to_time(start_dayobs) - TimeDelta(lead_days, format="jd"),
        rn_dayobs.day_obs_to_time(end_dayobs) + TimeDelta(1, format="jd"),
        efd_client,
    )
    # searchsorted below returns garbage, silently, on an unsorted index.
    return messages.sort_index()


def exclusive_cause(status_labels):
    """The single highest-priority cause in a pipe-joined label string.

    Label order varies in the EFD ("FAULT | WEATHER" and "WEATHER | IDLE" both
    occur), so this tests for substrings rather than comparing whole strings.
    """
    for cause in CAUSE_PRIORITY:
        if cause in status_labels:
            return cause
    return "UNKNOWN"


def parse_fault_cscs(note, regex=FAULT_CSC_RE):
    """Faulting CSC names (SAL index retained) from a status-message note, in note order.

    Two payload formats occur across 2026: the Scheduler emitted a Python-repr
    list ("state: ['MTMount', 'MTPtg'].") up to 2026-06-08 and a bare
    comma-separated list ("state: MTMount, MTPtg.") after. Strip the list
    punctuation so both yield the same names - otherwise the brackets and quotes
    become part of the name and one component splits into several phantom ones.
    """
    if not isinstance(note, str):
        return []
    names = []
    for match in regex.finditer(note):
        payload = match.group(1).strip().strip("[]")
        # Split on "," only - never on ":", which carries the SAL index.
        for name in payload.split(","):
            name = name.strip().strip("'\"").strip()
            if name and name not in names:
                names.append(name)
    # A single clause can repeat a CSC ("Scheduler:1, MTAOS, Scheduler:1, ...").
    return names


def csc_subsystem(csc):
    return CSC_SUBSYSTEMS.get(csc.split(":")[0], UNATTRIBUTED)


def override_cause(category, states, planned):
    """The budget cause of an override row, from its ROSP-24 `category` (B1 / C1 / C4 / C5 / D4).

    B1 is PLANNED_DOWNTIME, any C is WEATHER and D4 is DOWNTIME. A file without a category column (older
    versions) is reduced from its states by the same precedence: planned DOWNTIME, then WEATHER, then DOWNTIME.
    """
    if isinstance(category, str) and category:
        return {"B": "PLANNED_DOWNTIME", "C": "WEATHER", "D": "DOWNTIME"}.get(
            category[0], "UNKNOWN"
        )
    labels = {label.strip() for label in states.split("|")}
    if planned and "DOWNTIME" in labels:
        return "PLANNED_DOWNTIME"
    if "WEATHER" in labels:
        return "WEATHER"
    return exclusive_cause(states)


def overrides_location():
    """(url, commit SHA) of the override file: main pinned to the last commit that touched it, unless
    ROSP_STATE_OVERRIDES_CSV / STATE_OVERRIDES_URL is set."""
    if OVERRIDES_SOURCE:
        return OVERRIDES_SOURCE, "n/a (location set by environment)"
    api = f"https://api.github.com/repos/{OVERRIDES_REPO}/commits?sha=main&path={OVERRIDES_PATH}&per_page=1"
    try:
        with urllib.request.urlopen(api, timeout=30) as response:
            sha = json.loads(response.read())[0]["sha"]
        return (
            f"https://raw.githubusercontent.com/{OVERRIDES_REPO}/{sha}/{OVERRIDES_PATH}",
            sha,
        )
    except Exception as err:
        print(
            f"WARNING: could not resolve the override commit ({err}); reading the tip of main"
        )
        return (
            f"https://raw.githubusercontent.com/{OVERRIDES_REPO}/main/{OVERRIDES_PATH}",
            "UNRESOLVED",
        )


def load_state_overrides():
    """The ROSP-42 override table indexed by day_obs, with a reduced `cause` and its ROSP-24 `category`.

    Returns an empty table, with a warning, if the file cannot be read, so the
    notebook still runs on EFD state alone.
    """
    url, sha = overrides_location()
    try:
        table = pd.read_csv(url, dtype={"day_obs": int}, keep_default_na=False)
    except Exception as err:
        print(f"WARNING: no state overrides applied; could not read {url} ({err})")
        return pd.DataFrame(
            columns=["cause", "category", "status_labels", "source"]
        ).rename_axis("day_obs")
    planned = table.planned_downtime.astype(str).str.upper().eq("TRUE")
    if "category" not in table:
        table["category"] = ""
    table["cause"] = [
        override_cause(c, s, p)
        for c, s, p in zip(table.category, table.states, planned)
    ]
    # Same "A | B" form as EFD statusLabels, so substring tests treat both alike.
    table["status_labels"] = table.states.str.replace("|", " | ", regex=False)
    print(
        f"{len(table)} state overrides, {table.day_obs.min()}-{table.day_obs.max()}, from {url} (commit {sha}); "
        f"categories: {table.category.value_counts().to_dict()}"
    )
    return table.set_index("day_obs")


ROSP24_SHORT = {
    "PLANNED_DOWNTIME": "B1",
    "WEATHER": "C",
    "FAULT": "D1/D2",
    "DOWNTIME": "D4",
    "IDLE": "E",
    "OPERATIONAL": "A+B2+D3",
    "UNKNOWN": "F",
}

In [ ]:
# Everything below needs the EFD. get_clients() falls back rather than
# returning None when the EFD is unreachable, so guard on an empty result and
# let the notebook finish rather than failing the whole page.
day_obs_nights = rn_dayobs.day_obs_list(
    rn_dayobs.day_obs_to_time(start_dayobs),
    # day_obs_list is end-exclusive, so reach past end_dayobs to include it.
    rn_dayobs.day_obs_to_time(end_dayobs) + TimeDelta(1, format="jd"),
)

status_messages = fetch_status_messages(start_dayobs, end_dayobs, efd_client)
have_status = len(status_messages) > 0
state_overrides = load_state_overrides()

print(
    f"{len(day_obs_nights)} nights ({start_dayobs}-{end_dayobs}), "
    f"{len(status_messages)} observatory status messages"
)
if not have_status:
    print("No observatory status messages found - skipping the analysis below.")

In [ ]:
def build_night_windows(day_obs_list, sun_alt=-12):
    """-12 deg sunset/sunrise per day_obs, as TAI MJD floats.

    `rn_dayobs.day_obs_sunset_sunrise` is `lru_cache(maxsize=100)`, so a range
    longer than 100 nights thrashes the cache and re-runs astroplan on every
    call - ~9 s per full pass, several passes per notebook. Resolve each night
    once here and pass the result down.
    """
    windows = {}
    for day_obs in day_obs_list:
        sunset, sunrise = rn_dayobs.day_obs_sunset_sunrise(day_obs, sun_alt)
        windows[day_obs] = (sunset.tai.mjd, sunrise.tai.mjd)
    return windows


def build_state_intervals(
    messages,
    day_obs_list,
    night_windows,
    clamp_now=True,
    max_age_hours=MAX_STATE_AGE_HOURS,
    overrides=None,
):
    """Partition each night into non-overlapping observatory-state intervals.

    Returns one row per state run per night, clipped to -12 deg sunset/sunrise,
    with exactly one `cause` each - so hours per night sum to night_hours.
    A message's state is trusted for at most `max_age_hours`; time past that,
    or before the first message, is UNKNOWN with provenance "no_telemetry".
    A night in `overrides` is one interval with the override's cause, in place
    of whatever the EFD recorded.
    """
    # Work in TAI MJD floats throughout: the EFD index is tz-aware pandas,
    # which does not compare cleanly against astropy Time.
    msg_mjd = Time(messages.index.to_pydatetime(), scale="utc").tai.mjd
    labels = messages.statusLabels.astype(str).values
    notes = messages.note.values
    now_mjd = Time.now().tai.mjd
    max_age = max_age_hours / 24.0

    rows = []

    def add(day_obs, start, end, cause, label, note, provenance, msg=-1):
        if end > start:
            rows.append(
                {
                    "msg_index": msg,
                    "day_obs": day_obs,
                    "start_mjd": start,
                    "end_mjd": end,
                    "hours": (end - start) * 24,
                    "cause": cause,
                    "status_labels": label,
                    "note": note,
                    "provenance": provenance,
                }
            )

    for day_obs in day_obs_list:
        lo, hi = night_windows[day_obs]
        if clamp_now:
            if lo >= now_mjd:
                continue  # night has not started yet
            hi = min(hi, now_mjd)  # night in progress: count only elapsed time

        if overrides is not None and day_obs in overrides.index:
            row = overrides.loc[day_obs]
            add(day_obs, lo, hi, row.cause, row.status_labels, None, "override")
            continue

        # Seed from the last message at or before sunset, then take every
        # message falling inside the night.
        first = max(np.searchsorted(msg_mjd, lo, side="right") - 1, 0)
        last = np.searchsorted(msg_mjd, hi, side="left")
        idx = list(range(first, last))

        # No state known before the first usable message. Record it rather than
        # emitting nothing, which would let the night vanish from the budget
        # while the closure check trivially passed.
        edge = hi if len(idx) == 0 else max(min(msg_mjd[idx[0]], hi), lo)
        add(day_obs, lo, edge, "UNKNOWN", "", None, "no_telemetry")

        for i in idx:
            start = max(msg_mjd[i], lo)
            end = min(msg_mjd[i + 1], hi) if i + 1 < len(msg_mjd) else hi
            fresh_end = min(end, msg_mjd[i] + max_age)
            add(
                day_obs,
                start,
                fresh_end,
                exclusive_cause(labels[i]),
                labels[i],
                notes[i],
                "efd",
                i,
            )
            add(
                day_obs, max(start, fresh_end), end, "UNKNOWN", "", None, "no_telemetry"
            )

    intervals = (
        pd.DataFrame(rows).sort_values(["day_obs", "start_mjd"]).reset_index(drop=True)
    )
    intervals["measured"] = intervals.provenance == "efd"
    # Consecutive FAULT intervals form one physical fault run. Break runs at a
    # night boundary so a fault straddling sunrise is attributed to both nights.
    is_fault = intervals["status_labels"].str.contains("FAULT")
    new_run = (is_fault != is_fault.shift()) | (
        intervals["day_obs"] != intervals["day_obs"].shift()
    )
    intervals["is_fault"] = is_fault
    intervals["fault_run"] = new_run.cumsum().where(is_fault)
    return intervals


def night_budget_table(intervals, day_obs_list, night_windows, clamp_now=True):
    """Per-night hours by cause, complete over the whole calendar range.

    Indexed on every night in the range, not just nights that have visits -
    exactly the nights lost to a full-night fault or to weather have none,
    and those are the ones this notebook exists to show.
    """
    budget = intervals.pivot_table(
        index="day_obs", columns="cause", values="hours", aggfunc="sum", fill_value=0.0
    )
    budget = budget.reindex(
        index=day_obs_list, columns=list(BUDGET_CAUSES), fill_value=0.0
    ).fillna(0.0)

    now_mjd = Time.now().tai.mjd
    night_hours, day_obs_mjd = [], []
    for day_obs in budget.index:
        lo, hi = night_windows[day_obs]
        if clamp_now:
            hi = min(hi, now_mjd)
        night_hours.append(max(hi - lo, 0.0) * 24)
        day_obs_mjd.append(dayObsToTime(day_obs).mjd)
    budget["night_hours"] = night_hours
    budget["day_obs_mjd"] = day_obs_mjd
    return budget


if have_status:
    night_windows = build_night_windows(day_obs_nights)
    intervals = build_state_intervals(
        status_messages, day_obs_nights, night_windows, overrides=state_overrides
    )
    night_budget = night_budget_table(intervals, day_obs_nights, night_windows)

    # Per-night hours with fresh EFD telemetry, and hours set by an override.
    night_budget["efd_hours"] = (
        intervals[intervals.measured]
        .groupby("day_obs")
        .hours.sum()
        .reindex(night_budget.index, fill_value=0.0)
    )
    night_budget["override_hours"] = (
        intervals[intervals.provenance == "override"]
        .groupby("day_obs")
        .hours.sum()
        .reindex(night_budget.index, fill_value=0.0)
    )
    night_budget["efd_covered"] = (night_budget.night_hours > 0) & (
        night_budget.efd_hours >= MIN_COVERED_FRACTION * night_budget.night_hours
    )
    # Where the hours budget comes from. An override replaces the EFD state, so
    # it wins even where fresh EFD telemetry also exists.
    night_budget["budget_source"] = np.select(
        [night_budget.override_hours > 0, night_budget.efd_covered],
        ["override", "efd"],
        default="none",
    )

    # ROSP-24 category of each override night (B1 / C1 / C4 / C5 / D4).
    intervals["override_category"] = np.where(
        intervals.provenance == "override",
        intervals.day_obs.map(state_overrides.category).fillna(""),
        "",
    )
    night_budget["override_category"] = (
        state_overrides.category.reindex(night_budget.index).fillna("")
        if len(state_overrides)
        else ""
    )

    # ROSP-49: D2 is a whole-night fault - no survey visit, and the FAULT state (under any
    # cause) covers the night to within D2_TOLERANCE_MIN. Every other FAULT hour is D1.
    survey_nights = set(
        visits.loc[visits.science_program.isin(SURVEY_PROGRAMS), "day_obs"]
    )
    fault_label_hours = (
        intervals[intervals.is_fault]
        .groupby("day_obs")
        .hours.sum()
        .reindex(night_budget.index, fill_value=0.0)
    )
    night_budget["d2_night"] = (
        ~night_budget.index.isin(survey_nights)
        & (fault_label_hours > 0)
        & (fault_label_hours >= night_budget.night_hours - D2_TOLERANCE_MIN / 60.0)
    )
    night_budget["D2"] = np.where(night_budget.d2_night, night_budget.FAULT, 0.0)
    night_budget["D1"] = night_budget.FAULT - night_budget.D2

    # The partition must account for every hour of every night; if it does not,
    # every number below is wrong, so fail loudly rather than plotting it.
    closure = intervals.groupby("day_obs").hours.sum() - night_budget["night_hours"]
    assert np.abs(closure.dropna()).max() < 1e-4, closure
    print(f"partition closes to {np.abs(closure.dropna()).max():.2e} hr on all nights")

    stale = intervals[~intervals.measured]
    print(
        f"\nEFD-covered nights (>= {MIN_COVERED_FRACTION:.0%} fresh telemetry): "
        f"{int(night_budget.efd_covered.sum())} of {len(night_budget)}; "
        f"{stale.hours.sum():.1f} hr of night has no fresh telemetry "
        f"(state older than {MAX_STATE_AGE_HOURS:.0f} hr, or before the topic began)."
    )
    overridden = intervals[intervals.provenance == "override"]
    if len(overridden) > 0:
        print(
            f"\nState overrides set {overridden.hours.sum():.1f} hr on "
            f"{overridden.day_obs.nunique()} night(s), by cause:"
        )
        print(overridden.groupby("cause").hours.sum().round(1).to_string())
        # Weather the override records under a higher-precedence cause (planned
        # downtime): kept for statistics, never charged to WEATHER.
        shadow = overridden[
            overridden.status_labels.str.contains("WEATHER")
            & (overridden.cause != "WEATHER")
        ]
        if len(shadow) > 0:
            print(
                f"shadow record: {shadow.hours.sum():.1f} hr of weather on "
                f"{shadow.day_obs.nunique()} night(s) under "
                f"{', '.join(sorted(shadow.cause.unique()))}, not charged to WEATHER"
            )

#### Nights without EFD telemetry

`lsst.sal.Scheduler.logevent_observatoryStatus` does not extend indefinitely into the past —
it begins **2026-02-25** — and has gaps (for example after the July 2026 storm). Nights with
neither fresh telemetry nor an override are **unmeasured**: their time stays `UNKNOWN`, which is
ROSP-24 category F (unattributed), rather than being guessed.

Under ROSP-24 the narrative log is an **annotation** source: it confirms or corrects the
attribution but does not set hours. Earlier versions of this notebook backfilled unmeasured nights
from its `time_lost` entries; that is no longer done. The cell below still reads the log, reports
what it says about the unmeasured nights (as annotation, in the `log_fault` / `log_weather`
columns), and calibrates it against the EFD on nights where both exist, which shows why it cannot
carry hours: operators enter `time_lost` free-hand, entries overlap and sometimes exceed the night,
and weather closures in particular go unlogged.

In [ ]:
def narrative_time_lost(start_dayobs, end_dayobs, narrative_client, night_hours):
    """Per-night fault/weather hours reported by hand in the narrative log.

    Returns a frame indexed like `night_hours` with `log_fault`, `log_weather`,
    and `log_clipped` (hours discarded because the reported total exceeded the
    night). Operators log `time_lost` free-hand: entries overlap and some cover
    daytime recovery, so the raw sum can exceed the night several times over.
    """
    empty = pd.DataFrame(
        0.0,
        index=night_hours.index,
        columns=["log_fault", "log_weather", "log_clipped"],
    )
    logs = narrative_client.query_log(
        rn_dayobs.day_obs_to_time(start_dayobs),
        rn_dayobs.day_obs_to_time(end_dayobs) + TimeDelta(1, format="jd"),
        {"min_time_lost": "0.00001"},
    )
    if len(logs) == 0:
        return empty
    logs = logs.query("not component.str.contains('AuxTel')")
    if len(logs) == 0:
        return empty

    def to_day_obs(x):
        return rn_dayobs.day_obs_str_to_int(
            rn_dayobs.time_to_day_obs(Time(x.date_begin, format="isot", scale="utc"))
        )

    logs = logs.copy()
    logs["day_obs"] = logs.apply(to_day_obs, axis=1)
    # Data-quality counts, reported by the caller rather than hardcoded.
    n_zero_span = int(
        (
            pd.to_datetime(logs.date_end, utc=True, format="ISO8601")
            <= pd.to_datetime(logs.date_begin, utc=True, format="ISO8601")
        ).sum()
    )
    structured = [c for c in ("systems", "subsystems", "cscs") if c in logs.columns]
    n_structured = int(
        logs[structured]
        .apply(
            lambda col: col.map(
                lambda v: len(v) > 0 if hasattr(v, "__len__") else False
            )
        )
        .any(axis=1)
        .sum()
        if structured
        else 0
    )
    raw = (
        logs.pivot_table(
            index="day_obs",
            columns="time_lost_type",
            values="time_lost",
            aggfunc="sum",
            fill_value=0.0,
        )
        .reindex(night_hours.index, fill_value=0.0)
        .fillna(0.0)
    )
    for column in ("fault", "weather"):
        if column not in raw:
            raw[column] = 0.0
    raw = raw[["fault", "weather"]]

    # Scale each night down so reported loss cannot exceed the night length.
    total = raw.sum(axis=1)
    scale = pd.Series(
        np.where(total > night_hours, night_hours / total.replace(0.0, np.nan), 1.0),
        index=raw.index,
    ).fillna(1.0)
    scaled = raw.mul(scale, axis=0)
    out = pd.DataFrame(
        {
            "log_fault": scaled.fault,
            "log_weather": scaled.weather,
            "log_clipped": (total - night_hours).clip(lower=0.0),
        }
    )
    out.attrs = {
        "n_entries": len(logs),
        "n_zero_span": n_zero_span,
        "n_structured": n_structured,
        "max_ratio": float((total / night_hours.replace(0.0, np.nan)).max()),
    }
    return out


if have_status:
    narrative_lost = narrative_time_lost(
        start_dayobs, end_dayobs, clients["narrative_log"], night_budget.night_hours
    )
    night_budget = night_budget.join(narrative_lost)
    log_stats = narrative_lost.attrs
    if log_stats:
        print(
            f"Narrative log: {log_stats['n_entries']} Simonyi time-loss entries; "
            f"{log_stats['n_zero_span']} have a zero-length date span, "
            f"{log_stats['n_structured']} carry structured systems/subsystems/cscs, "
            f"largest reported loss is {log_stats['max_ratio']:.1f}x its night.\n"
        )

    # ROSP-24: the narrative log annotates, it does not set hours. Nights with neither
    # telemetry nor an override stay UNKNOWN (F); what the log says about them is reported.
    unmeasured = night_budget.budget_source == "none"
    night_budget["unmeasured"] = unmeasured
    if unmeasured.any():
        print(
            f"{int(unmeasured.sum())} night(s) without telemetry or an override "
            f"({night_budget.loc[unmeasured, 'night_hours'].sum():.1f} night hr) stay UNKNOWN (F). "
            f"The narrative log annotates {night_budget.loc[unmeasured, 'log_fault'].sum():.1f} hr of fault and "
            f"{night_budget.loc[unmeasured, 'log_weather'].sum():.1f} hr of weather on them; "
            "this is not added to the budget."
        )

    # Calibrate the log against the EFD where both exist, to show why it cannot carry hours.
    both = night_budget[night_budget.budget_source == "efd"]
    if len(both) > 0 and both.log_fault.sum() > 0:
        print(
            f"\nCalibration on the {len(both)} nights whose budget is EFD-derived "
            "(not overridden) - narrative log recovers:"
        )
        print(
            f"  fault:   {both.log_fault.sum():.1f} / {both.FAULT.sum():.1f} hr "
            f"= {both.log_fault.sum() / max(both.FAULT.sum(), 1e-9):.0%}"
        )
        print(
            f"  weather: {both.log_weather.sum():.1f} / {both.WEATHER.sum():.1f} hr "
            f"= {both.log_weather.sum() / max(both.WEATHER.sum(), 1e-9):.0%}"
        )

    # Nothing is backfilled, so the timeline and the event table use the partition itself.
    timeline_intervals = intervals

In [ ]:
if have_status:
    display(
        night_budget[
            list(BUDGET_CAUSES)
            + [
                "D1",
                "D2",
                "night_hours",
                "efd_covered",
                "budget_source",
                "override_category",
                "log_fault",
                "log_weather",
            ]
        ].round(2)
    )

    sources = night_budget.groupby("budget_source").night_hours.agg(["count", "sum"])
    print("Where each night's hours budget comes from:")
    for source, row in sources.iterrows():
        print(
            f"  {source:<10} {int(row['count']):4d} nights  {row['sum']:7.1f} night hr"
        )

    # Quote totals over nights whose budget is measured or overridden. Unmeasured
    # nights are ROSP-24 F and would dilute the loss fractions.
    accounted = night_budget[night_budget.budget_source.isin(["efd", "override"])]
    total_hours = accounted.night_hours.sum()
    print(
        f"\nTotals over the {len(accounted)} nights with an EFD or override "
        f"budget ({total_hours:.1f} night hr):"
    )
    totals = accounted[list(BUDGET_CAUSES)].sum()
    display(
        pd.DataFrame(
            {
                "ROSP-24": [ROSP24_CATEGORY[c] for c in totals.index],
                "hours": totals.round(2).values,
                "fraction": (totals / max(total_hours, 1e-9)).round(3).values,
            },
            index=totals.index,
        )
    )
    print(
        f"FAULT split per ROSP-49: D1 {accounted.D1.sum():.2f} hr, D2 (whole-night) "
        f"{accounted.D2.sum():.2f} hr on {int(accounted.d2_night.sum())} night(s) "
        f"{list(accounted.index[accounted.d2_night])}"
    )
    by_category = (
        accounted.groupby("override_category")[list(BUDGET_CAUSES)].sum().sum(axis=1)
    )
    if len(by_category) > 1:
        print("hours by override category ('' = EFD):", by_category.round(1).to_dict())
    lost = accounted[list(LOST_CAUSES)].sum().sum()
    if total_hours > 0:
        print(
            f"\noperational: {accounted.OPERATIONAL.sum():.1f} hr "
            f"({accounted.OPERATIONAL.sum() / total_hours:.0%})  "
            f"lost: {lost:.1f} hr ({lost / total_hours:.0%})  "
            f"unknown: {accounted.UNKNOWN.sum():.1f} hr"
        )
    if night_budget.unmeasured.any():
        print(
            "\nThe remaining nights have no telemetry and no override: they are ROSP-24 F "
            "(unattributed) and are left out of these totals."
        )

### Where the night went

Each bar is one night of $-12^\circ$ twilight-to-twilight time, partitioned into the causes
above; the black tick marks the night's full length, so a bar reaching it means every hour is
accounted for. Blue is time the scheduler considered itself operational — everything else is
lost time, and the colour says to what; the legend gives each cause's ROSP-24 category. Hours set by a state override are dotted. The x-axis covers every night in the range, including
nights with no visits at all.

In [ ]:
if have_status:
    # One x position per night in the range, not per night with visits:
    # dropping the visit-less nights would hide the largest losses.
    xticks_budget = night_budget["day_obs_mjd"].values
    xtick_labels_budget = night_budget.index.values

    # A label per night is unreadable much past a fortnight, so thin them to
    # roughly 25 across whatever range was requested. Shared by every plot below.
    tick_step = slice(None, None, max(len(xticks_budget) // 25, 1))
    bar_width = 0.8 if len(xticks_budget) <= 40 else 1.0

    plt.figure(figsize=(16, 8), dpi=200)
    bottom = np.zeros(len(night_budget))
    for cause in BUDGET_CAUSES:
        hours = night_budget[cause].values
        if hours.sum() <= 0:
            continue
        curated = night_budget.budget_source.values == "override"
        plt.bar(
            xticks_budget,
            hours,
            bottom=bottom,
            width=bar_width,
            color=CAUSE_COLORS[cause],
            hatch=[".." if c else None for c in curated],
            edgecolor="0.35",
            linewidth=0.0,
            label=f"{cause} = {ROSP24_SHORT[cause]} ({hours.sum():.1f} hr)",
        )
        bottom += hours

    # The stack must equal the night.
    assert np.abs(bottom - night_budget["night_hours"].values).max() < 1e-4

    plt.plot(
        xticks_budget,
        night_budget["night_hours"].values,
        "k_",
        ms=18 if len(xticks_budget) <= 40 else 4,
        label="Night length (-12 deg)",
    )
    plt.ylabel("Hours of -12 deg Night")
    plt.xticks(
        xticks_budget[tick_step],
        xtick_labels_budget[tick_step],
        rotation=90.0,
        fontsize=8,
    )
    plt.xlim(xticks_budget[0] - 1.0, xticks_budget[-1] + 1.0)
    # Legend above the axes, so the y-range only needs to clear the longest night.
    plt.legend(
        loc="lower left",
        bbox_to_anchor=(0.0, 1.01),
        fontsize=7,
        ncol=5,
        frameon=False,
    )
    plt.ylim(0.0, night_budget["night_hours"].max() * 1.05)
    plt.minorticks_off()
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig("figures/night_budget_%i-%i.pdf" % (start_dayobs, end_dayobs))
    plt.show()

In [ ]:
# Messages as arrays, for walking back from a fault run to the clause that started it.
msg_mjd_all = Time(status_messages.index.to_pydatetime(), scale="utc").tai.mjd
msg_labels_all = status_messages.statusLabels.astype(str).values
msg_are_all = [parse_fault_cscs(n) for n in status_messages.note.values]


def run_onset(group):
    """The CSCs behind a fault run, where they were found, and the message time (TAI MJD).

    In order: the run's first message; for a fault carried across twilight, the earliest
    "are in" clause of the unbroken FAULT stretch the run continues; for a FAULT set by hand
    with no clause, the last clause within FAULT_ONSET_LOOKBACK_HOURS; a later message of the
    run; the sticky "were in" clause.
    """
    first = int(group.msg_index.iloc[0])
    if first >= 0 and msg_are_all[first]:
        return msg_are_all[first], "run start", msg_mjd_all[first]
    if first >= 0:
        j, found = first - 1, None
        while j >= 0 and "FAULT" in msg_labels_all[j]:
            if msg_are_all[j]:
                found = j
            j -= 1
        if found is not None:
            return msg_are_all[found], "FAULT stretch", msg_mjd_all[found]
        j = first - 1
        while (
            j >= 0
            and msg_mjd_all[j] >= msg_mjd_all[first] - FAULT_ONSET_LOOKBACK_HOURS / 24
        ):
            if msg_are_all[j]:
                return msg_are_all[j], "lookback", msg_mjd_all[j]
            j -= 1
    for note, start in zip(group.note, group.start_mjd):
        if parse_fault_cscs(note):
            return parse_fault_cscs(note), "later in run", start
    for note, start in zip(group.note, group.start_mjd):
        if parse_fault_cscs(note, FAULT_CSC_WERE_RE):
            return parse_fault_cscs(note, FAULT_CSC_WERE_RE), "were in", start
    return [], "none", np.nan


def first_to_fault(cscs, t_onset, tolerance_s=120.0):
    """Of several CSCs named at once, the one whose summaryState entered FAULT (3) first."""
    best, best_t = None, np.inf
    t0 = Time(t_onset - FAULT_ONSET_LOOKBACK_HOURS / 24, format="mjd", scale="tai")
    t1 = Time(t_onset + tolerance_s / 86400, format="mjd", scale="tai")
    for csc in cscs:
        base, _, index = csc.partition(":")
        try:
            states = efd_client.select_time_series(
                f"lsst.sal.{base}.logevent_summaryState",
                ["summaryState"],
                t0,
                t1,
                index=int(index) if index else None,
            )
        except Exception:
            continue
        faults = states[states.summaryState == 3] if len(states) else states
        if len(faults):
            t = Time(faults.index[-1].to_pydatetime(), scale="utc").tai.mjd
            if t < best_t:
                best, best_t = csc, t
    return best


def fault_runs_table(intervals):
    """One row per fault run, charged to its primary CSC (ROSP-24: the first CSC to enter FAULT).

    `hours` is the run's full extent; `hours_charged` counts only the part the
    budget charges to FAULT, i.e. excluding time the run spent inside a WEATHER period.
    """
    records = []
    for run_id, group in intervals.dropna(subset=["fault_run"]).groupby("fault_run"):
        onset, source, t_onset = run_onset(group)
        if len(onset) > 1:
            primary = first_to_fault(onset, t_onset) or onset[0]
            rule = "first to enter FAULT (summaryState)" if primary else "first named"
        else:
            primary, rule = (onset[0] if onset else None), "only CSC named"
        named = sorted(
            set().union(*(set(parse_fault_cscs(n)) for n in group.note)) | set(onset)
        )
        records.append(
            {
                "day_obs": int(group.day_obs.iloc[0]),
                "fault_run": int(run_id),
                "start_mjd": group.start_mjd.min(),
                "hours": group.hours.sum(),
                "hours_charged": group.loc[group.cause == "FAULT", "hours"].sum(),
                "csc": primary or UNATTRIBUTED,
                "subsystem": csc_subsystem(primary) if primary else UNATTRIBUTED,
                "onset_source": source,
                "primary_rule": rule if primary else "no CSC found",
                "n_cscs": len(named),
                "cscs_named": ", ".join(named),
                "note": group.note.iloc[0],
            }
        )
    return pd.DataFrame(records)


if have_status:
    fault_runs = fault_runs_table(intervals)

    # The charged hours must be exactly the FAULT column of the budget.
    assert abs(fault_runs.hours_charged.sum() - night_budget.FAULT.sum()) < 1e-6

    fault_by_subsystem = (
        fault_runs.groupby("subsystem").hours_charged.sum().sort_values(ascending=False)
    )
    # Fault that ran under a higher-priority cause (WEATHER), by that cause: the ROSP-24
    # shadow record. Only fresh telemetry can carry a FAULT label, so nothing here is
    # carried across a gap.
    fault_hidden_by_cause = (
        intervals.query("is_fault and cause != 'FAULT'").groupby("cause").hours.sum()
    )
    fault_hidden = fault_hidden_by_cause.sum()

    print("Fault hours charged, by ROSP-24 attribution group of the primary CSC:")
    print(fault_by_subsystem.round(2).to_string())
    print("\nFault hours charged, by primary CSC:")
    print(
        fault_runs.groupby("csc")
        .hours_charged.sum()
        .sort_values(ascending=False)
        .round(2)
        .to_string()
    )
    print(
        "\nWhere each run's CSC came from:",
        fault_runs.onset_source.value_counts().to_dict(),
    )
    print(
        f"\nShadow record: a further {fault_hidden:.2f} hr of fault ran while the observatory was "
        "already closed for weather; those hours are charged to weather, not to "
        "fault, and are not counted above:"
    )
    if len(fault_hidden_by_cause):
        print(fault_hidden_by_cause.round(2).to_string())
    multi = fault_runs.query("n_cscs > 1")
    if len(multi) > 0:
        print(
            f"{len(multi)} fault run(s) named more than one CSC; each is charged whole to its primary CSC."
        )

### Fault time by subsystem

Fault hours attributed to the ROSP-24 group of each run's primary CSC — per night on the left,
totalled over the range on the right. Only the hours the budget charges to
`FAULT` appear here; fault time that ran underneath a weather or downtime period is reported
above instead, so this plot does not blame the system for time already lost to weather. Bars
labelled `Pending attribution` are fault runs for which no CSC clause was found at all.

In [ ]:
if have_status:
    subsystem_by_night = (
        fault_runs.pivot_table(
            index="day_obs",
            columns="subsystem",
            values="hours_charged",
            aggfunc="sum",
            fill_value=0.0,
        )
        .reindex(index=night_budget.index, fill_value=0.0)
        .fillna(0.0)
    )
    ordered = fault_by_subsystem[fault_by_subsystem > 0]

    fig, axes = plt.subplots(
        1, 2, figsize=(18, 6), dpi=200, gridspec_kw={"width_ratios": [2.2, 1]}
    )

    ax = axes[0]
    bottom = np.zeros(len(subsystem_by_night))
    for subsystem in ordered.index:
        hours = subsystem_by_night[subsystem].values
        ax.bar(
            xticks_budget,
            hours,
            bottom=bottom,
            width=bar_width,
            color=SUBSYSTEM_COLORS.get(subsystem, "0.6"),
            label=subsystem,
        )
        bottom += hours
    ax.set_ylabel("Fault Hours")
    ax.set_xticks(xticks_budget[tick_step])
    ax.set_xticklabels(xtick_labels_budget[tick_step], rotation=90.0, fontsize=8)
    ax.set_xlim(xticks_budget[0] - 1.0, xticks_budget[-1] + 1.0)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.3, axis="y")
    ax.minorticks_off()

    ax = axes[1]
    positions = np.arange(len(ordered))[::-1]
    ax.barh(
        positions,
        ordered.values,
        color=[SUBSYSTEM_COLORS.get(s, "0.6") for s in ordered.index],
    )
    for position, hours in zip(positions, ordered.values):
        ax.text(hours, position, "  %.2f" % hours, va="center", fontsize=9)
    ax.set_yticks(positions)
    ax.set_yticklabels(ordered.index, fontsize=9)
    ax.set_xlabel("Fault Hours (%i-%i)" % (start_dayobs, end_dayobs))
    ax.set_xlim(0.0, ordered.max() * 1.25)
    ax.grid(alpha=0.3, axis="x")
    ax.minorticks_off()

    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig("figures/fault_by_subsystem_%i-%i.pdf" % (start_dayobs, end_dayobs))
    plt.show()

### Fault rate and recovery time per component

The budget above says how many hours each subsystem cost; this says how those hours were
made up — whether a subsystem is failing *often* or failing *slowly*. Two quantities per
component:

- **Fault rate**, faults per 100 hr of *exposure*, about ten nights of observing at the mean
  -12° night of 10 hr. Exposure is `OPERATIONAL + FAULT`, i.e.
  time the observatory was actually trying to observe. Hours lost to weather or downtime are
  excluded from the denominator, and — for consistency — fault runs that occurred *entirely*
  within a weather or downtime period are excluded from the numerator too. Counting those
  faults against an exposure that omits their hours would inflate the rate with failures that
  cost no observing time.
- **Recovery time**, the wall-clock duration of each fault run. Note this is the run's *full*
  extent, not the `hours_charged` used in the budget: how long the component took to recover
  is an engineering property of the fault, whereas the charged hours are a time-accounting
  one, and the two differ whenever a fault straddled the start or end of a weather period.

Each fault run is one event, attributed to its primary CSC as in the budget, so the event counts
sum to the number of fault runs. Runs are broken at the end of the night, so a recovery time is
censored at the night's end (ROSP-49).

**Read the rates as small-number statistics.** Over a short range, or one with little
observing, most components fault only a handful of times; error bars are $\sqrt{N}$ (Poisson) and the lower arm is clipped at
zero, and a component with `N=1` has no meaningful uncertainty at all. Medians are quoted in
preference to means because the distribution is heavily skewed — a few long faults
dominate any mean.

In [ ]:
def fault_events_table(fault_runs, exposed_only=True):
    """One row per fault run, attributed to its primary CSC, for rate/MTTR statistics.

    The recovery time is the run's full duration. With exposed_only, a fault run that ran
    entirely inside a WEATHER period is dropped: its hours are excluded from the exposure
    denominator, so it is excluded from the numerator as well.
    """
    events = fault_runs.copy()
    if exposed_only:
        events = events[events.hours_charged > 1e-9]
    events = events.rename(
        columns={"hours": "recovery_hours", "hours_charged": "charged_hours"}
    )
    return events[
        [
            "day_obs",
            "fault_run",
            "start_mjd",
            "recovery_hours",
            "charged_hours",
            "csc",
            "subsystem",
        ]
    ].reset_index(drop=True)


# Fault rates are quoted per RATE_PER_HOURS of exposure (OPERATIONAL + FAULT).
RATE_PER_HOURS = 100.0


def reliability_stats(fault_events, exposure_hours, level="subsystem"):
    """Fault rate per RATE_PER_HOURS of exposure, and recovery-time summary, by `level`."""
    stats = fault_events.groupby(level).agg(
        n_faults=("fault_run", "nunique"),
        median_recovery=("recovery_hours", "median"),
        mean_recovery=("recovery_hours", "mean"),
        max_recovery=("recovery_hours", "max"),
    )
    stats["rate"] = stats.n_faults / exposure_hours * RATE_PER_HOURS
    # Poisson sqrt(N); symmetric, so the lower arm is clipped when plotted.
    stats["rate_err"] = np.sqrt(stats.n_faults) / exposure_hours * RATE_PER_HOURS
    stats["mtbf_hours"] = exposure_hours / stats.n_faults
    return stats.sort_values("n_faults", ascending=False)


if have_status:
    fault_events = fault_events_table(fault_runs)

    # Exposure: hours the observatory was actually trying to observe. Weather and
    # downtime hours are excluded, matching the numerator above. Restricted to
    # EFD-covered nights - a night with no telemetry contributes no fault events,
    # so including its hours would silently depress every rate.
    exposure = (night_budget.OPERATIONAL + night_budget.FAULT).where(
        night_budget.efd_covered, 0.0
    )
    exposure_hours = exposure.sum()

    subsystem_reliability = reliability_stats(fault_events, exposure_hours, "subsystem")
    csc_reliability = reliability_stats(fault_events, exposure_hours, "csc")

    n_events = fault_events.fault_run.nunique()
    n_excluded = (
        int(intervals.fault_run.nunique()) - n_events
        if intervals.fault_run.notna().any()
        else 0
    )
    print(
        f"exposure (OPERATIONAL+FAULT): {exposure_hours:.2f} hr over "
        f"{(exposure > 0).sum()} EFD-covered night(s)"
    )
    print(
        f"{n_events} fault runs counted "
        f"({n_excluded} excluded as entirely within weather or downtime)"
    )
    print(
        f"overall: {n_events / exposure_hours * RATE_PER_HOURS:.2f} faults per "
        f"{RATE_PER_HOURS:.0f} hr of exposure, "
        f"MTBF {exposure_hours / n_events:.2f} hr, "
        f"median recovery {fault_events.recovery_hours.median() * 60:.1f} min"
    )
    print("\nBy subsystem:")
    display(
        subsystem_reliability[
            [
                "n_faults",
                "rate",
                "mtbf_hours",
                "median_recovery",
                "mean_recovery",
                "max_recovery",
            ]
        ].round(3)
    )
    print("By CSC:")
    display(
        csc_reliability[
            ["n_faults", "rate", "mtbf_hours", "median_recovery", "mean_recovery"]
        ].round(3)
    )

In [ ]:
if have_status and len(fault_events) > 0:
    ordered_rel = subsystem_reliability.index
    positions = np.arange(len(ordered_rel))[::-1]

    fig, axes = plt.subplots(1, 2, figsize=(17, 6), dpi=200)

    ax = axes[0]
    rate = subsystem_reliability.rate.values
    rate_err = subsystem_reliability.rate_err.values
    ax.barh(
        positions,
        rate,
        color=[SUBSYSTEM_COLORS.get(s, "0.6") for s in ordered_rel],
    )
    # Clip the lower arm at the bar: sqrt(N) is symmetric and would run negative
    # at low N, which is meaningless for a rate.
    ax.errorbar(
        rate,
        positions,
        xerr=[np.minimum(rate_err, rate), rate_err],
        fmt="none",
        ecolor="0.25",
        elinewidth=1.2,
        capsize=3,
    )
    for position, subsystem in zip(positions, ordered_rel):
        n_faults = subsystem_reliability.loc[subsystem, "n_faults"]
        ax.text(
            subsystem_reliability.loc[subsystem, "rate"]
            + subsystem_reliability.loc[subsystem, "rate_err"],
            position,
            "  N=%i" % n_faults,
            va="center",
            fontsize=9,
        )
    ax.set_yticks(positions)
    ax.set_yticklabels(ordered_rel, fontsize=9)
    ax.set_xlabel("Faults per %.0f hr of Exposure" % RATE_PER_HOURS)
    ax.set_xlim(0.0, (rate + rate_err).max() * 1.28)
    ax.grid(alpha=0.3, axis="x")
    ax.minorticks_off()

    ax = axes[1]
    for i, subsystem in enumerate(ordered_rel):
        recovery = np.sort(
            fault_events.loc[
                fault_events.subsystem == subsystem, "recovery_hours"
            ].values
            * 60
        )
        y = np.full(len(recovery), len(ordered_rel) - 1 - i, dtype=float)
        if len(recovery) > 1:
            # Spread ties vertically so overlapping points stay countable.
            y += np.linspace(-0.17, 0.17, len(recovery))
        ax.scatter(
            recovery,
            y,
            s=44,
            color=SUBSYSTEM_COLORS.get(subsystem, "0.6"),
            edgecolor="0.25",
            lw=0.5,
            zorder=3,
        )
        median = np.median(recovery)
        ax.plot(
            [median, median],
            [len(ordered_rel) - 1 - i - 0.33, len(ordered_rel) - 1 - i + 0.33],
            color="black",
            lw=2.4,
            zorder=4,
            label="median" if i == 0 else None,
        )
    ax.set_yticks(positions)
    ax.set_yticklabels(ordered_rel, fontsize=9)
    ax.set_xscale("log")
    ax.set_xlabel("Recovery Time per Fault (minutes)")
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.3, axis="x")

    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(
            "figures/fault_rate_recovery_%i-%i.pdf" % (start_dayobs, end_dayobs)
        )
    plt.show()

### Fault rate and recovery time over the range

Top: every fault as a point at its night, coloured by subsystem, with recovery time on a log
axis — the spread can be several orders of magnitude, from second-long glitches to
full-night faults. The dashed line is the running median over all components, which is
the only trend line the statistics support: per-night counts for any single subsystem are
0–5, so a per-subsystem recovery trend would be noise.

Bottom: cumulative fault rate, i.e. faults so far divided by exposure so far. Reading it
left to right shows whether a component's rate is settling down or climbing as the range
progresses; the early nights are volatile because both numerator and denominator are small.
Components with a single fault are omitted, having no meaningful cumulative rate.

In [ ]:
if have_status and len(fault_events) > 0:
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(16, 10),
        dpi=200,
        sharex=True,
        gridspec_kw={"height_ratios": [1.35, 1]},
    )

    ax = axes[0]
    for subsystem in ordered_rel:
        selection = fault_events[fault_events.subsystem == subsystem].sort_values(
            "day_obs"
        )
        x = np.array([dayObsToTime(d).mjd for d in selection.day_obs], dtype=float)
        # Spread same-night points deterministically, so re-running the notebook
        # reproduces the figure exactly (no RNG).
        for night in np.unique(x):
            same_night = x == night
            if same_night.sum() > 1:
                x[same_night] += np.linspace(-0.3, 0.3, same_night.sum())
        ax.scatter(
            x,
            selection.recovery_hours * 60,
            s=54,
            color=SUBSYSTEM_COLORS.get(subsystem, "0.6"),
            edgecolor="0.25",
            lw=0.5,
            label="%s (N=%i)"
            % (subsystem, subsystem_reliability.loc[subsystem, "n_faults"]),
            zorder=3,
        )
    ordered_events = fault_events.sort_values("start_mjd")
    ax.plot(
        [dayObsToTime(d).mjd for d in ordered_events.day_obs],
        ordered_events.recovery_hours.expanding().median() * 60,
        color="black",
        lw=1.8,
        ls="--",
        zorder=4,
        label="running median (all)",
    )
    ax.set_yscale("log")
    ax.set_ylabel("Recovery Time (min)")
    ax.legend(loc="upper left", fontsize=8, ncol=3)
    ax.grid(alpha=0.3)

    ax = axes[1]
    faults_per_night = fault_events.pivot_table(
        index="day_obs",
        columns="subsystem",
        values="fault_run",
        aggfunc="nunique",
        fill_value=0,
    ).reindex(night_budget.index, fill_value=0)
    cumulative_exposure = exposure.cumsum()
    for subsystem in ordered_rel:
        if subsystem_reliability.loc[subsystem, "n_faults"] < 2:
            continue  # a single fault gives no meaningful cumulative rate
        with np.errstate(divide="ignore", invalid="ignore"):
            cumulative_rate = np.where(
                cumulative_exposure.values > 0,
                faults_per_night[subsystem].cumsum().values
                / cumulative_exposure.values
                * RATE_PER_HOURS,
                np.nan,
            )
        ax.plot(
            xticks_budget,
            cumulative_rate,
            marker="o",
            ms=4,
            color=SUBSYSTEM_COLORS.get(subsystem, "0.6"),
            label=subsystem,
        )
    overall_rate = fault_events.fault_run.nunique() / exposure_hours * RATE_PER_HOURS
    ax.axhline(
        overall_rate,
        color="black",
        ls=":",
        lw=1.5,
        label="all components (%.1f / %.0f hr)" % (overall_rate, RATE_PER_HOURS),
    )
    ax.set_ylabel("Cumulative Faults / %.0f hr of Exposure" % RATE_PER_HOURS)
    ax.set_xticks(xticks_budget[tick_step])
    ax.set_xticklabels(xtick_labels_budget[tick_step], rotation=90.0, fontsize=8)
    ax.set_xlim(xticks_budget[0] - 1.0, xticks_budget[-1] + 1.0)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.3)
    ax.minorticks_off()

    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig("figures/fault_rate_trend_%i-%i.pdf" % (start_dayobs, end_dayobs))
    plt.show()

### Rolling fault rate and recovery time

With enough nights the per-fault scatter above becomes hard to read, and a running median
anchored at the start of the range stops responding to change. These panels use a centred
**28-night rolling window** instead, which is long enough that most subsystems accumulate
several faults in it and short enough to follow real change.

Top: rolling fault rate per subsystem — faults in the window divided by exposure in the same
window, so nights lost to weather do not depress the rate. Only subsystems with enough
events over the whole range are drawn. Bottom: rolling median recovery time over all
components, with the interquartile band, on a log axis.

Both are computed over EFD-covered nights only. A window overlapping the pre-telemetry
period is not plotted rather than being scaled up from partial data.

In [ ]:
ROLLING_NIGHTS = 28
MIN_FAULTS_TO_TREND = 5

if have_status and len(fault_events) > 0:
    covered_index = night_budget.index[night_budget.efd_covered]
    # Exposure per night, restricted to covered nights - a window that reaches
    # into the pre-telemetry period would otherwise divide by near-zero.
    exposure_covered = exposure.reindex(covered_index).fillna(0.0)
    faults_covered = (
        fault_events.pivot_table(
            index="day_obs",
            columns="subsystem",
            values="fault_run",
            aggfunc="nunique",
            fill_value=0,
        )
        .reindex(covered_index, fill_value=0)
        .fillna(0)
    )

    window = min(ROLLING_NIGHTS, max(len(covered_index) // 3, 3))
    roll_exposure = exposure_covered.rolling(
        window, center=True, min_periods=window
    ).sum()
    x_covered = np.array([dayObsToTime(d).mjd for d in covered_index])

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(16, 10),
        dpi=200,
        sharex=True,
        gridspec_kw={"height_ratios": [1, 1]},
    )

    ax = axes[0]
    trended = [
        s
        for s in subsystem_reliability.index
        if subsystem_reliability.loc[s, "n_faults"] >= MIN_FAULTS_TO_TREND
        and s in faults_covered.columns
    ]
    for subsystem in trended:
        roll_faults = (
            faults_covered[subsystem]
            .rolling(window, center=True, min_periods=window)
            .sum()
        )
        with np.errstate(divide="ignore", invalid="ignore"):
            roll_rate = np.where(
                roll_exposure > 0,
                roll_faults / roll_exposure * RATE_PER_HOURS,
                np.nan,
            )
        ax.plot(
            x_covered,
            roll_rate,
            lw=2.0,
            color=SUBSYSTEM_COLORS.get(subsystem, "0.6"),
            label="%s (N=%i)"
            % (subsystem, subsystem_reliability.loc[subsystem, "n_faults"]),
        )
    total_roll = (
        faults_covered.sum(axis=1)
        .rolling(window, center=True, min_periods=window)
        .sum()
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        ax.plot(
            x_covered,
            np.where(
                roll_exposure > 0,
                total_roll / roll_exposure * RATE_PER_HOURS,
                np.nan,
            ),
            lw=1.6,
            ls="--",
            color="black",
            label="all components",
        )
    ax.set_ylabel(
        "Faults / %.0f hr of Exposure (%i-night window)" % (RATE_PER_HOURS, window)
    )
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    ax.grid(alpha=0.3)
    ax.minorticks_off()

    ax = axes[1]
    # Median recovery per night, then a rolling median of the individual faults:
    # reindex events onto nights so the window is in nights, not in events.
    per_night_events = fault_events.groupby("day_obs").recovery_hours.apply(list)
    windowed_median, windowed_lo, windowed_hi = [], [], []
    half = window // 2
    for i, day_obs in enumerate(covered_index):
        lo_i, hi_i = max(i - half, 0), min(i + half + 1, len(covered_index))
        if hi_i - lo_i < window:
            windowed_median.append(np.nan)
            windowed_lo.append(np.nan)
            windowed_hi.append(np.nan)
            continue
        pool = [
            v
            for d in covered_index[lo_i:hi_i]
            if d in per_night_events.index
            for v in per_night_events[d]
        ]
        if len(pool) == 0:
            windowed_median.append(np.nan)
            windowed_lo.append(np.nan)
            windowed_hi.append(np.nan)
        else:
            windowed_median.append(np.median(pool) * 60)
            windowed_lo.append(np.percentile(pool, 25) * 60)
            windowed_hi.append(np.percentile(pool, 75) * 60)
    ax.fill_between(
        x_covered,
        windowed_lo,
        windowed_hi,
        color="tab:blue",
        alpha=0.2,
        label="25th-75th percentile",
    )
    ax.plot(x_covered, windowed_median, lw=2.0, color="tab:blue", label="median")
    ax.set_yscale("log")
    ax.set_ylabel("Recovery Time (min, %i-night window)" % window)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.3)

    ax.set_xticks(xticks_budget[tick_step])
    ax.set_xticklabels(xtick_labels_budget[tick_step], rotation=90.0, fontsize=8)
    ax.set_xlim(xticks_budget[0] - 1.0, xticks_budget[-1] + 1.0)

    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig("figures/fault_rate_rolling_%i-%i.pdf" % (start_dayobs, end_dayobs))
    plt.show()

### Visit timeline against observatory state

One column per night, with the time of night on the y-axis. The background is shaded by the
cause in force at each moment and the black ticks are survey visits, so a gap between visits
can be read off against what the observatory was doing. This is the only view in the
notebook that is resolved *within* the night: the budget above gives each night's totals, this
shows when they happened. A column shaded solid with no ticks is a night lost outright.

Nights without EFD telemetry are `UNKNOWN` (grey, ROSP-24 F); the narrative log is not used to
fill them. Solid shading is EFD-derived and time-resolved; dotted shading was set by a state
override (ROSP-42).

The shading spans $-12^\circ$ twilight only, from the same `rubin_nights` night boundaries as
the budget.

In [ ]:
if have_status:
    plt.figure(figsize=(16, 8), dpi=200)

    # Shade each state interval in place (timeline_intervals is the partition itself).
    shade_width = 1.0 if len(xticks_budget) > 40 else 0.9
    for interval in timeline_intervals.itertuples():
        reference = dayObsToTime(interval.day_obs)
        y_start = convertTime(
            Time(interval.start_mjd, format="mjd", scale="tai"), reference
        )
        y_end = convertTime(
            Time(interval.end_mjd, format="mjd", scale="tai"), reference
        )
        backfilled = interval.provenance == "narrative"
        curated = interval.provenance == "override"
        plt.bar(
            reference.mjd,
            y_end - y_start,
            bottom=y_start,
            width=shade_width,
            color=CAUSE_COLORS[interval.cause],
            alpha=0.55,
            linewidth=0.0,
            # Hatched where the hours come from the narrative log: the log gives
            # only a night total, so the amount of colour is real but its
            # vertical placement is arbitrary.
            hatch="///" if backfilled else (".." if curated else None),
            edgecolor="0.35" if (backfilled or curated) else None,
        )
    # Legend entries only - the bars above are drawn per interval.
    efd_only = timeline_intervals.query("provenance != 'narrative'")
    for cause in BUDGET_CAUSES:
        if efd_only.query("cause == @cause").hours.sum() > 0:
            plt.bar(np.nan, 0.0, color=CAUSE_COLORS[cause], alpha=0.55, label=cause)
    if (timeline_intervals.provenance == "narrative").any():
        plt.bar(
            np.nan,
            0.0,
            color="0.5",
            alpha=0.55,
            hatch="///",
            edgecolor="0.35",
            label="narrative log (night total only)",
        )
    if (timeline_intervals.provenance == "override").any():
        plt.bar(
            np.nan,
            0.0,
            color="0.5",
            alpha=0.55,
            hatch="..",
            edgecolor="0.35",
            label="state override (ROSP-42)",
        )

    survey_days = np.unique(visits["day_obs"])
    for day_obs in survey_days:
        reference = dayObsToTime(day_obs)
        selection_day_obs = visits["day_obs"] == day_obs
        selection_fbs = np.isin(
            visits[selection_day_obs]["science_program"], SURVEY_PROGRAMS
        )
        y = convertTime(
            Time(visits["exp_midpt_mjd"][selection_day_obs], format="mjd"), reference
        )
        plt.scatter(
            np.tile(reference.mjd, np.sum(selection_fbs)),
            y[selection_fbs],
            marker="_",
            s=20,
            c="black",
            zorder=5,
            label="Survey Programs" if day_obs == survey_days[0] else None,
        )

    plt.legend(loc="upper left", fontsize=8, markerscale=2, ncol=3)
    plt.xlim(xticks_budget[0] - 1.0, xticks_budget[-1] + 1.0)
    plt.ylim(-1.5, 11.0)
    plt.xticks(
        xticks_budget[tick_step],
        xtick_labels_budget[tick_step],
        rotation=90.0,
        fontsize=8,
    )
    plt.ylabel("Time (hrs)")
    plt.minorticks_off()
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig("figures/state_timeline_%i-%i.pdf" % (start_dayobs, end_dayobs))
    plt.show()

### Individual lost-time events

The largest losses, each attributed to a cause. Two kinds of row appear: `visit gap` for a
gap longer than five minutes between consecutive visits, and `no visits` for a night that
produced none at all — those are seeded explicitly, since by construction they generate no
gaps and would otherwise be missing from exactly the nights that matter most.

A long gap can span several state intervals, so `cause` is the state holding the most hours
of the gap rather than whichever happened to be in force at its first instant. `fault_cscs`
lists every CSC recorded as faulted anywhere within the span — populated even when the cause
is weather, since a fault can run underneath it. `note` is the operator/scheduler text, kept
raw so that annotations like intentional engineering faults remain visible.

In [ ]:
def attribute_span(start_mjd, end_mjd, intervals):
    """Dominant cause over [start_mjd, end_mjd], plus CSCs faulted within it."""
    overlap = (
        np.minimum(intervals.end_mjd.values, end_mjd)
        - np.maximum(intervals.start_mjd.values, start_mjd)
    ).clip(min=0)
    if overlap.sum() <= 0:
        # Entirely outside -12 deg night, i.e. observing in twilight, which the
        # partition does not cover. Distinct from the EFD's own UNKNOWN state.
        return pd.Series({"cause": "TWILIGHT", "fault_cscs": "", "note": None})
    covered = intervals[overlap > 0]
    hours = pd.Series(overlap[overlap > 0] * 24, index=covered.index)
    cause = hours.groupby(covered.cause).sum().idxmax()
    cscs = sorted(set().union(*(set(parse_fault_cscs(n)) for n in covered.note), set()))
    dominant = covered[covered.cause == cause]
    note = next((n for n in dominant.note if isinstance(n, str) and n.strip()), None)
    return pd.Series({"cause": cause, "fault_cscs": ",".join(cscs), "note": note})


def lost_time_events(visits, intervals, night_budget, min_gap_s=300.0):
    """Visit gaps over min_gap_s, plus whole nights that produced no visits."""
    rows = []
    for day_obs, group in visits.groupby("day_obs"):
        group = group.sort_values("obs_start_mjd")
        gap_start = group.obs_end_mjd.values[:-1]
        gap_end = group.obs_start_mjd.values[1:]
        gap_seconds = (gap_end - gap_start) * 86400
        for i in np.where(gap_seconds > min_gap_s)[0]:
            rows.append(
                {
                    "day_obs": day_obs,
                    "kind": "visit gap",
                    "start_mjd": gap_start[i],
                    "end_mjd": gap_end[i],
                    "hours": gap_seconds[i] / 3600,
                }
            )
    nights_with_visits = set(visits["day_obs"])
    for day_obs in night_budget.index:
        if day_obs not in nights_with_visits:
            sunset, sunrise = rn_dayobs.day_obs_sunset_sunrise(day_obs, -12)
            rows.append(
                {
                    "day_obs": day_obs,
                    "kind": "no visits",
                    "start_mjd": sunset.tai.mjd,
                    "end_mjd": sunrise.tai.mjd,
                    "hours": night_budget.night_hours[day_obs],
                }
            )
    if len(rows) == 0:
        return pd.DataFrame(
            columns=["day_obs", "kind", "hours", "cause", "fault_cscs", "note"]
        )
    events = pd.DataFrame(rows)
    attribution = events.apply(
        lambda x: attribute_span(x.start_mjd, x.end_mjd, intervals), axis=1
    )
    return events.join(attribution).sort_values(
        "hours", ascending=False, ignore_index=True
    )


if have_status:
    # A gap on a night without telemetry is UNKNOWN (F): the narrative log is annotation only.
    events = lost_time_events(visits, timeline_intervals, night_budget)
    print(
        f"{len(events)} events totalling {events.hours.sum():.1f} hr "
        f"({len(events.query('kind == \"no visits\"'))} nights with no visits)"
    )
    print("\nHours by attributed cause:")
    print(
        events.groupby("cause")
        .hours.sum()
        .sort_values(ascending=False)
        .round(2)
        .to_string()
    )
    with pd.option_context("display.max_colwidth", 90):
        display(
            events[["day_obs", "kind", "hours", "cause", "fault_cscs", "note"]]
            .head(25)
            .round(2)
        )

### Cross-checks against the other accountings

Two other sources describe the same lost time, and neither agrees with the partition for
reasons that are worth making explicit rather than leaving as a third number that differs.
The `rubin_nights` observatory states are compared column by column — their periods overlap,
so they are an upper bound on each cause and must never be summed — and the narrative log is
what the observing specialists reported by hand (an annotation source under ROSP-24).

In [ ]:
if have_status:
    # 1. rubin_nights observatory states. These periods OVERLAP and must not be
    #    summed, so each is compared column by column, not totalled together.
    #    They are upper bounds on the partition: a night-spanning WEATHER period
    #    also covers the OPERATIONAL/FAULT periods inside it.
    state_periods = observatory_status.get_observatory_state_times(
        rn_dayobs.day_obs_to_time(start_dayobs),
        rn_dayobs.day_obs_to_time(end_dayobs) + TimeDelta(1, format="jd"),
        efd_client,
    )
    if len(state_periods) > 0:
        dome_open = observatory_status.get_dome_open_close(
            rn_dayobs.day_obs_to_time(start_dayobs),
            rn_dayobs.day_obs_to_time(end_dayobs) + TimeDelta(1, format="jd"),
            efd_client,
        )
        state_periods, state_summary = observatory_status.count_observatory_states(
            state_periods, dome_open
        )
        comparison = pd.DataFrame(
            {
                "weather (partition)": night_budget["WEATHER"],
                "weather_down (states)": state_summary["weather_down"],
                "fault (partition)": night_budget["FAULT"],
                "fault_down (states)": state_summary["fault_down"],
            }
        ).fillna(0.0)
        display(comparison.round(2))
        print(comparison.sum().round(2).to_string())
        diff = (
            comparison["fault_down (states)"].sum()
            - comparison["fault (partition)"].sum()
        )
        under_weather = fault_hidden_by_cause.get("WEATHER", 0.0)
        under_downtime = intervals.query(
            "is_fault and cause == 'FAULT' and status_labels.str.contains('DOWNTIME')"
        ).hours.sum()
        print(
            f"\nfault_down - fault = {diff:.2f} hr. count_observatory_states removes FAULT that "
            "overlaps DOWNTIME, which the ROSP-24 partition charges to the fault, and keeps FAULT under "
            "WEATHER, which the partition charges to weather; so this is expected to be about "
            f"{under_weather:.2f} hr (fault under weather) - {under_downtime:.2f} hr (fault under "
            f"DOWNTIME) = {under_weather - under_downtime:.2f} hr."
        )
        print(
            "rubin_nights carries the last state across telemetry gaps, which the "
            f"partition treats as unmeasured after {MAX_STATE_AGE_HOURS:.0f} hr, so "
            "the two can also differ over gaps."
        )

    # 2. The narrative log, i.e. what the observing specialists reported by hand.
    #    Expect this to be much smaller: it is only filled in when someone
    #    writes up a time-loss entry, and its structured subsystem fields are
    #    generally empty. Under ROSP-24 it is annotation only, never hours.
    time_lost_logs = clients["narrative_log"].query_log(
        rn_dayobs.day_obs_to_time(start_dayobs),
        rn_dayobs.day_obs_to_time(end_dayobs) + TimeDelta(1, format="jd"),
        {"min_time_lost": "0.00001"},
    )
    if len(time_lost_logs) > 0:
        time_lost_logs = time_lost_logs.query("not component.str.contains('AuxTel')")
    if len(time_lost_logs) > 0:
        print("\nNarrative log (Simonyi only), hours reported lost by type:")
        print(
            time_lost_logs.groupby("time_lost_type")
            .time_lost.sum()
            .round(2)
            .to_string()
        )
        print(
            f"({len(time_lost_logs)} entries; compare "
            f"{night_budget.FAULT.sum():.1f} hr fault and "
            f"{night_budget.WEATHER.sum():.1f} hr weather from the EFD partition)"
        )
    else:
        print("\nNo Simonyi narrative-log time-loss entries in this range.")

# Summary

This notebook answers *why* on-sky time went missing, by partitioning every night — including
nights that produced no visits at all, which views built from the gaps between visits cannot
see — into operational time versus time lost to weather,
downtime, or a fault, and attributing the fault hours to a subsystem. Read the accountings
together rather than interchangeably:

- The **partition** (`night_budget`) is exhaustive per night and mutually exclusive by
  construction, so its columns can be summed and compared against `night_hours`. Its
  reduction order is the ROSP-24 precedence as far as the EFD states can express it, and each
  cause maps to a ROSP-24 category (`ROSP24_CATEGORY`); the official ledger,
  `rosp-11_time_loss_accounting`, adds the splits this notebook does not make (A1–A3, B2a/B2b,
  D3, C2/C3).
- The **`rubin_nights` observatory states** cross-check is an upper bound on each cause, since
  those periods overlap; the difference from the partition is the fault time that ran while
  the observatory was already down for weather or downtime.
- The **narrative log** is what observing specialists reported by hand. Under ROSP-24 it is an
  annotation source: on nights with telemetry it corroborates the totals, on nights without it
  is shown but not counted (those nights stay `UNKNOWN`, category F). The calibration printed
  above shows how much of the EFD fault and weather time it recovers.
- **State overrides** ([ROSP-42](https://rubinobs.atlassian.net/browse/ROSP-42)) supply the
  state where the EFD did not record it, such as the July–October 2026 off-sky period. They
  are read from the shared file on `main` at a pinned commit, take their ROSP-24 `category`,
  and are dotted in the plots; weather under planned downtime is reported as the shadow record
  only.

The fault-rate and recovery-time views then split those fault hours into *how often* versus
*how slowly* each component failed — the distinction that says whether a subsystem needs
better reliability or faster recovery. A subsystem can dominate the hours budget either way,
and the two point at different work. Over a long range the rolling panels are the ones to
read for trend; the per-fault scatter is for spotting individual events.

Caveats worth restating:

- Telemetry coverage is not the same as the requested range. A state older than
  `MAX_STATE_AGE_HOURS` is treated as unmeasured (F), not carried forward. Quote loss
  percentages over the nights with an EFD or override budget.
- `FAULT` includes intentional engineering faults (ROSP-49 Q8: any FAULT is a fault), and the
  attributed subsystem is the CSC that entered `FAULT` first — the component, not necessarily
  the root cause; the operator note can say otherwise (a `FAULT` held by hand for a different
  subsystem keeps the CSC of the clause that started it).
- The hours budget and the reliability statistics now use the same attribution: each fault run
  goes whole to its primary CSC, so `fault_runs` reconciles with `night_budget` and the event
  counts in `fault_events` sum to the number of runs.